# IS-Bench: Colab simulator setup

Run these cells **top to bottom in a fresh Google Colab runtime** with an **L4 GPU and High-RAM** selected. This recreates the working Python 3.10, CUDA PyTorch, Vulkan, Isaac Sim 4.1, and OmniGibson 1.1.1 setup. Runtime files are temporary. A normal disconnect may preserve the VM, but idle expiry or “Disconnect and delete runtime” removes its installed environment and data.

This notebook downloads OmniGibson's basic assets. It does **not** download the BEHAVIOR scene/object dataset or the IS-Bench scene archive; those are the next phase. It also does not start Isaac Sim, so there is no shutdown smoke test here.

The NVIDIA driver-matching step below was verified on the Colab driver `580.82.07`. It detects the active driver and stops with a clear error if the corresponding Ubuntu package is unavailable, rather than installing a mismatched graphics library.


## 1. Check the Colab runtime

In [ ]:
import shutil
import subprocess
from pathlib import Path

def run(*args, env=None):
    print("Running:", " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), check=True, env=env)

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU is attached. Select a GPU runtime in Colab and reconnect.")
run("nvidia-smi")
run("free", "-h")
run("df", "-h", "/content")


## 2. Install system libraries and match Vulkan to Colab's NVIDIA driver

In [ ]:
run("apt-get", "update", "-qq")
run("apt-get", "install", "-y", "--no-install-recommends",
    "vulkan-tools", "libvulkan1", "libglu1-mesa")

driver = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    text=True,
).splitlines()[0].strip()
major = driver.split(".")[0]
package = f"libnvidia-gl-{major}"
package_list = subprocess.check_output(["apt-cache", "madison", package], text=True)
versions = [line.split("|")[1].strip() for line in package_list.splitlines() if "|" in line]
matches = [version for version in versions if version.split("-")[0] == driver]
if not matches:
    raise RuntimeError(
        f"Colab driver {driver} has no matching {package} in APT. "
        "Stop here and inspect the new runtime; do not install a different driver version."
    )
apt_version = matches[0]

preference = Path("/tmp/isbench-nvidia.pref")
preference.write_text(
    f"Package: src:nvidia-graphics-drivers-{major}:any\n"
    f"Pin: version {apt_version}\n"
    "Pin-Priority: 1001\n"
)
print(f"Matching graphics package: {package}={apt_version}")
run("apt-get", "install", "-y",
    "-o", f"Dir::Etc::preferences={preference}",
    "--no-install-recommends", f"{package}={apt_version}")


In [ ]:
import os

vulkan_env = os.environ.copy()
vulkan_env["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
result = subprocess.run(
    ["vulkaninfo", "--summary"], env=vulkan_env,
    text=True, capture_output=True, check=True,
)
print(result.stdout)
if "deviceName" not in result.stdout or "NVIDIA" not in result.stdout:
    raise RuntimeError("Vulkan did not enumerate the NVIDIA GPU.")
print("VULKAN_READY")


## 3. Create the Python 3.10 environment and install CUDA PyTorch

In [ ]:
conda = Path("/content/miniforge3/bin/conda")
python = Path("/content/miniforge3/envs/isbench-gpu/bin/python")
if not conda.exists():
    installer = Path("/content/Miniforge3.sh")
    run("curl", "-fL",
        "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
        "-o", installer)
    run("bash", installer, "-b", "-p", "/content/miniforge3")
if not python.exists():
    run(conda, "create", "-y", "-n", "isbench-gpu", "python=3.10", "pip", "numpy<2")

run(python, "-m", "pip", "install",
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu121")
run(python, "-c", "import torch; print(torch.__version__, torch.version.cuda, "
    "torch.cuda.is_available(), torch.cuda.get_device_name(0) "
    "if torch.cuda.is_available() else 'no GPU'); "
    "assert torch.version.cuda is not None and torch.cuda.is_available()")


## 4. Install OmniGibson and Isaac Sim

In [ ]:
run(python, "-m", "pip", "install", "omnigibson==1.1.1",
    "--index-url", "https://pypi.org/simple")

install_env = vulkan_env.copy()
install_env["OMNI_KIT_ACCEPT_EULA"] = "YES"
run(conda, "run", "--no-capture-output", "-n", "isbench-gpu",
    "python", "-m", "omnigibson.install", "--no-install-datasets",
    env=install_env)


## 5. Apply the Isaac Sim compatibility fixes and download basic assets

In [ ]:
# Isaac Sim 4.1 loads an older bundled botocore; these matching versions
# avoid the DEFAULT_CHECKSUM_ALGORITHM import error observed in Colab.
run(python, "-m", "pip", "install",
    "boto3==1.28.61", "botocore==1.31.61", "s3transfer==0.7.0")

# Only OmniGibson's basic assets (including sky.jpg), not the large scene dataset.
run(python, "-c",
    "from omnigibson.utils.asset_utils import download_assets; download_assets()")


## 6. Lightweight verification (no simulator launch)

In [ ]:
check = r'''from importlib.metadata import version
from pathlib import Path
import torch
from omnigibson.macros import gm

assert torch.cuda.is_available(), "CUDA unavailable in isbench-gpu"
assert torch.version.cuda is not None, "CPU-only PyTorch installed"
assert version("omnigibson") == "1.1.1"
sky = Path(gm.ASSET_PATH) / "models/background/sky.jpg"
assert sky.is_file() and sky.stat().st_size > 0, f"Missing basic asset: {sky}"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("OmniGibson:", version("omnigibson"))
print("Isaac Sim:", version("isaacsim"))
print("Basic assets:", sky.parent)
print("SETUP_READY")'''
run(python, "-c", check)
